# VT-GraF Benchmark: Visible--Thermal Granular Faults under severe clutter
## Comparison: Unsupervised Generalized Frangi Graph vs. Standard Frangi + SAM 2

This notebook reproduces the comparison on the **VT-GraF-Dataset** (multimodal asphalt crack dataset with severe granular clutter) between:
1. **Ours (Generalized Frangi Graph)**: Our unsupervised multimodal graph-based method (using MST + Betweenness Centrality on GPU).
2. **Baseline (Standard Frangi on Thermal + SAM 2 on Visible)**: Zero-shot SAM 2 prompted automatically by standard Frangi filter responses from the thermal modality.

---
### Technical Setup:
- Automatically downloads the **VT-GraF-Dataset** from Google Drive.
- Performs an optimized sparse Git clone to check out only the required `ISPRS/` source folder (saving time and bandwidth).
- Installs **Segment Anything 2 (SAM 2)** from Meta's repository and downloads the `sam2_hiera_large` weights.
- Runs both methods and outputs comparative metrics: Jaccard (IoU), Tversky index, and Wasserstein distance.


## 1. Environment Setup & Datasets
We download the dataset, install SAM 2, and retrieve the pre-trained weights.


In [ ]:
# Install required packages
!pip install -q gdown POT scikit-image opencv-python matplotlib scipy torch torchvision

# Install SAM 2 from Meta's source
!pip install -q git+https://github.com/facebookresearch/sam2.git

# Download SAM 2 Large weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt



In [ ]:
from skimage.filters import frangi

def run_baseline_frangi_sam2(sample, predictor, num_prompts=12):
    # Retrieve the images
    img_vis_t = sample['visible'] # Torch tensor [0, 1]
    img_ir_t  = sample['infrared'] # Torch tensor [0, 1]
    
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_ir_np = (img_ir_t.numpy() * 255).astype(np.uint8)
    
    # 1. Apply standard Frangi on the thermal image
    # Note: We use sigmas=[20, 30, 40] to match our method's scales exactly
    frangi_response = frangi(img_ir_np / 255.0, sigmas=[20, 30, 40])
    
    # 2. Threshold the response to get a binary mask
    # We take the top 0.5% intensity pixels to represent the crack
    thresh = np.percentile(frangi_response, 99.5)
    binary_mask = frangi_response > thresh
    
    # 3. Skeletonize to get a thin centerline
    skel = skeletonize_lee(binary_mask)
    
    # 4. Sample point coordinates along the skeleton
    y_coords, x_coords = np.where(skel > 0)
    total_pts = len(y_coords)
    
    if total_pts < 3:
        # Fallback if the Frangi response is empty: sample from max intensity pixel
        y_max, x_max = np.unravel_index(np.argmax(frangi_response), frangi_response.shape)
        pts = np.array([[x_max, y_max]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
    else:
        # Regular spatial sampling
        step = max(1, total_pts // num_prompts)
        pts_x = x_coords[::step][:num_prompts]
        pts_y = y_coords[::step][:num_prompts]
        pts = np.column_stack((pts_x, pts_y)).astype(np.float32)
        labels = np.ones(len(pts), dtype=np.int32) # 1 means positive points
        
    # 5. Predict using SAM 2 on the visible image
    # Convert visible grayscale to RGB (3-channels) for SAM 2
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    predictor.set_image(img_vis_rgb)
    
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return {
        'pred_mask': pred_mask,
        'pts': pts,
        'frangi_response': frangi_response,
        'binary_mask': binary_mask,
        'skel': skel
    }


## 2. Load the Generalized Frangi Graph Modules
We perform an optimized shallow, sparse Git clone to fetch only the `ISPRS/` directory containing our custom GPU graph extraction code.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

results = []
visualizations = {}

# Set weights to match the latest notebook: 1/3 Visible, 2/3 Infrared
weights_ours = {'visible': 1/3, 'infrared': 2/3}

for i in range(len(dataset)):
    sample = dataset[i]
    name = sample['id']
    print(f"\nProcessing {name}...")
    
    # --- 1. RUN OURS (Generalized Frangi Graph) ---
    imgs_i = {'visible': sample['visible'], 'infrared': sample['infrared']}
    
    # Run the latest code parameters: K=2 and multiscale Σ=[20,30,40]
    max_S_global, sim_img, centrality_i, timings_i, diagnostics_i = extract_frangi_graph_gpu(
        imgs_i, weights_ours, 
        Σ=[20, 30, 40], R=3, K=2, device=device
    )
    
    pred_ours = (centrality_i > 0.025).astype(np.uint8)
    # Post-process: skeletonize and thicken to 5 pixels (matching latest notebook)
    sk_pred_ours = skeletonize_lee(pred_ours)
    sk_pred_thick_ours = thicken(sk_pred_ours, pixels=5)
    
    # --- 2. RUN BASELINE (Frangi Thermal + SAM 2) ---
    baseline_outputs = run_baseline_frangi_sam2(sample, predictor, num_prompts=12)
    pred_baseline = baseline_outputs['pred_mask']
    sampled_pts = baseline_outputs['pts']
    sk_pred_baseline = skeletonize_lee(pred_baseline)
    sk_pred_thick_baseline = thicken(sk_pred_baseline, pixels=5)
    
    # --- 3. GROUND TRUTH SKELETON ---
    gt_arr = sample['gt'].numpy().astype(np.uint8)
    sk_gt = skeletonize_lee(gt_arr)
    sk_gt_thick = thicken(sk_gt, pixels=5)
    
    # --- 4. COMPUTE METRICS ---
    # Ours
    jac_ours, tv_ours = compute_metrics(sk_pred_thick_ours, sk_gt_thick)
    wass_ours = wasserstein_distance_skeletons(sk_pred_thick_ours, sk_gt_thick)
    
    # Baseline
    jac_base, tv_base = compute_metrics(sk_pred_thick_baseline, sk_gt_thick)
    wass_base = wasserstein_distance_skeletons(sk_pred_thick_baseline, sk_gt_thick)
    
    results.append({
        'Fissure': name,
        'Ours_Jaccard': jac_ours,
        'Ours_Tversky': tv_ours,
        'Ours_Wasserstein': wass_ours,
        'Baseline_Jaccard': jac_base,
        'Baseline_Tversky': tv_base,
        'Baseline_Wasserstein': wass_base
    })
    
    visualizations[name] = {
        'visible': sample['visible'].numpy(),
        'infrared': sample['infrared'].numpy(),
        'gt': gt_arr,
        'ours_pred': pred_ours,
        'ours': sk_pred_thick_ours,
        'ours_centrality': centrality_i,
        'ours_fused_frangi': max_S_global,
        'ours_similarity': sim_img,
        'ours_tau_mask': diagnostics_i.get('tau_mask', np.zeros_like(pred_ours)),
        'ours_comp_mask': diagnostics_i.get('comp_mask', np.zeros_like(pred_ours)),
        'baseline': sk_pred_thick_baseline,
        'baseline_mask': pred_baseline,
        'points': sampled_pts,
        'baseline_frangi': baseline_outputs['frangi_response'],
        'baseline_binary': baseline_outputs['binary_mask'],
        'baseline_skel': baseline_outputs['skel']
    }

df_res = pd.DataFrame(results)


## 3. Initialize SAM 2 Predictor
We load the pre-trained SAM 2 Large model on GPU (or CPU as fallback) and initialize the image predictor.


In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

predictor = SAM2ImagePredictor(build_sam2(model_cfg, sam2_checkpoint, device=device))
print("SAM 2 Large predictor loaded successfully!")



for name in sorted(visualizations.keys()):
    vis_data = visualizations[name]
    
    # We create a large grid: 3 rows and 4 columns to display all steps!
    fig, axes = plt.subplots(3, 4, figsize=(28, 20))
    fig.suptitle(f"COMPREHENSIVE VISUAL COMPARISON - {name}", fontsize=22, fontweight='bold')
    
    # --- ROW 1: INPUT MODALITIES & GENERAL INPUT FRANGI ---
    # 1. Visible Image
    axes[0, 0].imshow(vis_data['visible'], cmap='gray')
    axes[0, 0].set_title("1. Visible Modality (Severe Gravel Clutter)", fontsize=14, fontweight='bold')
    
    # 2. Thermal Image
    axes[0, 1].imshow(vis_data['infrared'], cmap='gray')
    axes[0, 1].set_title("2. Decoded Inverted Thermal", fontsize=14, fontweight='bold')
    
    # 3. Ground Truth Mask
    axes[0, 2].imshow(vis_data['gt'], cmap='gray')
    axes[0, 2].set_title("3. Ground Truth (Fissure Target)", fontsize=14, fontweight='bold')
    
    # 4. Our Fused Multiscale Frangi
    axes[0, 3].imshow(vis_data['ours_fused_frangi'], cmap='magma')
    axes[0, 3].set_title("4. Ours: Fused Frangi Response (Hessian Level)", fontsize=14, fontweight='bold')
    
    # --- ROW 2: OURS (GENERALIZED FRANGI GRAPH ON GPU) ---
    # 5. Node Similarity Map
    axes[1, 0].imshow(vis_data['ours_similarity'], cmap='magma')
    axes[1, 0].set_title("5. Ours: Graph Node Similarities (Max)", fontsize=14, fontweight='bold')
    
    # 6. Graph Filtering Diagnostics
    h, w = vis_data['visible'].shape
    rgba_tau = np.zeros((h, w, 4))
    rgba_tau[vis_data['ours_tau_mask'] > 0] = [1.0, 1.0, 1.0, 0.3] # White for tau-filtered candidates
    rgba_comp = np.zeros((h, w, 4))
    rgba_comp[vis_data['ours_comp_mask'] > 0] = [0.0, 0.5, 1.0, 0.8] # Blue for large CCs
    axes[1, 1].imshow(np.zeros((h, w)), cmap='gray')
    axes[1, 1].imshow(rgba_tau)
    axes[1, 1].imshow(rgba_comp)
    axes[1, 1].set_title("6. Ours: Pruned Nodes (White) & Active CCs (Blue)", fontsize=14, fontweight='bold')
    
    # 7. Betweenness Centrality
    axes[1, 2].imshow(vis_data['ours_centrality'], cmap='hot')
    axes[1, 2].set_title("7. Ours: GPU Betweenness Centrality", fontsize=14, fontweight='bold')
    
    # 8. Ours final output
    axes[1, 3].imshow(vis_data['ours'], cmap='hot')
    axes[1, 3].set_title("8. Ours: Final Thickened Skeleton (5px)", fontsize=14, fontweight='bold')
    
    # --- ROW 3: BASELINE (STANDARD FRANGI + SAM 2) ---
    # 9. Baseline Standard Thermal Frangi
    axes[2, 0].imshow(vis_data['baseline_frangi'], cmap='magma')
    axes[2, 0].set_title("9. Baseline: Standard Thermal Frangi Response", fontsize=14, fontweight='bold')
    
    # 10. Top 0.5% Thresholded Mask
    axes[2, 1].imshow(vis_data['baseline_binary'], cmap='gray')
    axes[2, 1].set_title("10. Baseline: Top 0.5% Thresholded Mask", fontsize=14, fontweight='bold')
    
    # 11. Skeleton with 12 Prompt Points
    axes[2, 2].imshow(vis_data['baseline_skel'], cmap='gray')
    pts = vis_data['points']
    axes[2, 2].scatter(pts[:, 0], pts[:, 1], color='red', s=50, edgecolors='black', label='Sampled Prompts (N=12)')
    axes[2, 2].set_title("11. Baseline: Skeleton & Prompt Locations (Red)", fontsize=14, fontweight='bold')
    axes[2, 2].legend()
    
    # 12. Final Overlay comparison
    # White = Ground Truth, Green = Ours, Red = Baseline SAM 2
    rgba_gt = np.zeros((h, w, 4))
    rgba_gt[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.4] # White
    rgba_ours = np.zeros((h, w, 4))
    rgba_ours[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4] # Green
    rgba_base = np.zeros((h, w, 4))
    rgba_base[vis_data['baseline'] > 0] = [1.0, 0.0, 0.0, 0.4] # Red
    
    axes[2, 3].imshow(vis_data['visible'], cmap='gray')
    axes[2, 3].imshow(rgba_gt)
    axes[2, 3].imshow(rgba_ours)
    axes[2, 3].imshow(rgba_base)
    axes[2, 3].set_title("12. Overlay (White: GT, Green: Ours, Red: SAM 2)", fontsize=14, fontweight='bold')
    
    for ax in axes.flat:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()


In [ ]:
from skimage.filters import frangi

def run_baseline_frangi_sam2(sample, predictor, num_prompts=12):
    # Retrieve the images
    img_vis_t = sample['visible'] # Torch tensor [0, 1]
    img_ir_t  = sample['infrared'] # Torch tensor [0, 1]
    
    img_vis_np = (img_vis_t.numpy() * 255).astype(np.uint8)
    img_ir_np = (img_ir_t.numpy() * 255).astype(np.uint8)
    
    # 1. Apply standard Frangi on the thermal image
    # Note: We use sigmas=[20, 30, 40] to match our method's scales exactly
    frangi_response = frangi(img_ir_np / 255.0, sigmas=[20, 30, 40])
    
    # 2. Threshold the response to get a binary mask
    # We take the top 0.5% intensity pixels to represent the crack
    thresh = np.percentile(frangi_response, 99.5)
    binary_mask = frangi_response > thresh
    
    # 3. Skeletonize to get a thin centerline
    skel = skeletonize_lee(binary_mask)
    
    # 4. Sample point coordinates along the skeleton
    y_coords, x_coords = np.where(skel > 0)
    total_pts = len(y_coords)
    
    if total_pts < 3:
        # Fallback if the Frangi response is empty: sample from max intensity pixel
        y_max, x_max = np.unravel_index(np.argmax(frangi_response), frangi_response.shape)
        pts = np.array([[x_max, y_max]], dtype=np.float32)
        labels = np.array([1], dtype=np.int32)
    else:
        # Regular spatial sampling
        step = max(1, total_pts // num_prompts)
        pts_x = x_coords[::step][:num_prompts]
        pts_y = y_coords[::step][:num_prompts]
        pts = np.column_stack((pts_x, pts_y)).astype(np.float32)
        labels = np.ones(len(pts), dtype=np.int32) # 1 means positive points
        
    # 5. Predict using SAM 2 on the visible image
    # Convert visible grayscale to RGB (3-channels) for SAM 2
    img_vis_rgb = cv2.cvtColor(img_vis_np, cv2.COLOR_GRAY2RGB)
    predictor.set_image(img_vis_rgb)
    
    masks, scores, logits = predictor.predict(
        point_coords=pts,
        point_labels=labels,
        multimask_output=False
    )
    
    pred_mask = masks[0].astype(np.uint8)
    return {
        'pred_mask': pred_mask,
        'pts': pts,
        'frangi_response': frangi_response,
        'binary_mask': binary_mask,
        'skel': skel
    }


## 5. Quantitative Evaluation & Benchmark
We evaluate both methods on all 5 images of the VT-GraF-Dataset.
- For **Ours (Generalized Frangi Graph)**, we use parameters: `K=2` (dual triangle graph), scale set `Σ=[20,30,40]` and weights `visible: 1/3, infrared: 2/3` (the latest parameters from your notebook). Skeletons are thickened to **5 pixels** (`pixels=5`) for final evaluation.
- For the **Baseline (Frangi Thermal + SAM 2)**, we generate the mask, skeletonize it, and thicken it to **5 pixels** similarly.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

results = []
visualizations = {}

# Set weights to match the latest notebook: 1/3 Visible, 2/3 Infrared
weights_ours = {'visible': 1/3, 'infrared': 2/3}

for i in range(len(dataset)):
    sample = dataset[i]
    name = sample['id']
    print(f"\nProcessing {name}...")
    
    # --- 1. RUN OURS (Generalized Frangi Graph) ---
    imgs_i = {'visible': sample['visible'], 'infrared': sample['infrared']}
    
    # Run the latest code parameters: K=2 and multiscale Σ=[20,30,40]
    max_S_global, sim_img, centrality_i, timings_i, diagnostics_i = extract_frangi_graph_gpu(
        imgs_i, weights_ours, 
        Σ=[20, 30, 40], R=3, K=2, device=device
    )
    
    pred_ours = (centrality_i > 0.025).astype(np.uint8)
    # Post-process: skeletonize and thicken to 5 pixels (matching latest notebook)
    sk_pred_ours = skeletonize_lee(pred_ours)
    sk_pred_thick_ours = thicken(sk_pred_ours, pixels=5)
    
    # --- 2. RUN BASELINE (Frangi Thermal + SAM 2) ---
    baseline_outputs = run_baseline_frangi_sam2(sample, predictor, num_prompts=12)
    pred_baseline = baseline_outputs['pred_mask']
    sampled_pts = baseline_outputs['pts']
    sk_pred_baseline = skeletonize_lee(pred_baseline)
    sk_pred_thick_baseline = thicken(sk_pred_baseline, pixels=5)
    
    # --- 3. GROUND TRUTH SKELETON ---
    gt_arr = sample['gt'].numpy().astype(np.uint8)
    sk_gt = skeletonize_lee(gt_arr)
    sk_gt_thick = thicken(sk_gt, pixels=5)
    
    # --- 4. COMPUTE METRICS ---
    # Ours
    jac_ours, tv_ours = compute_metrics(sk_pred_thick_ours, sk_gt_thick)
    wass_ours = wasserstein_distance_skeletons(sk_pred_thick_ours, sk_gt_thick)
    
    # Baseline
    jac_base, tv_base = compute_metrics(sk_pred_thick_baseline, sk_gt_thick)
    wass_base = wasserstein_distance_skeletons(sk_pred_thick_baseline, sk_gt_thick)
    
    results.append({
        'Fissure': name,
        'Ours_Jaccard': jac_ours,
        'Ours_Tversky': tv_ours,
        'Ours_Wasserstein': wass_ours,
        'Baseline_Jaccard': jac_base,
        'Baseline_Tversky': tv_base,
        'Baseline_Wasserstein': wass_base
    })
    
    visualizations[name] = {
        'visible': sample['visible'].numpy(),
        'infrared': sample['infrared'].numpy(),
        'gt': gt_arr,
        'ours_pred': pred_ours,
        'ours': sk_pred_thick_ours,
        'ours_centrality': centrality_i,
        'ours_fused_frangi': max_S_global,
        'ours_similarity': sim_img,
        'ours_tau_mask': diagnostics_i.get('tau_mask', np.zeros_like(pred_ours)),
        'ours_comp_mask': diagnostics_i.get('comp_mask', np.zeros_like(pred_ours)),
        'baseline': sk_pred_thick_baseline,
        'baseline_mask': pred_baseline,
        'points': sampled_pts,
        'baseline_frangi': baseline_outputs['frangi_response'],
        'baseline_binary': baseline_outputs['binary_mask'],
        'baseline_skel': baseline_outputs['skel']
    }

df_res = pd.DataFrame(results)


## 6. Comparison Results Table


In [ ]:
# Format and display the results
pd.set_option('display.precision', 4)
display(df_res)

# Print Global Averages
print("\n" + "="*50)
print("--- SUMMARY STATISTICS ---")
print("="*50)
print(f"Jaccard (IoU) Mean   : Ours = {df_res['Ours_Jaccard'].mean():.4f} | Baseline = {df_res['Baseline_Jaccard'].mean():.4f}")
print(f"Tversky Index Mean   : Ours = {df_res['Ours_Tversky'].mean():.4f} | Baseline = {df_res['Baseline_Tversky'].mean():.4f}")
print(f"Wasserstein Dist Mean: Ours = {df_res['Ours_Wasserstein'].mean():.4f} | Baseline = {df_res['Baseline_Wasserstein'].mean():.4f}")



## 7. Visual Inspection
We plot the results side-by-side to visually analyze the structural output.


In [ ]:
for name in sorted(visualizations.keys()):
    vis_data = visualizations[name]
    
    # We create a large grid: 3 rows and 4 columns to display all steps!
    fig, axes = plt.subplots(3, 4, figsize=(28, 20))
    fig.suptitle(f"COMPREHENSIVE VISUAL COMPARISON - {name}", fontsize=22, fontweight='bold')
    
    # --- ROW 1: INPUT MODALITIES & GENERAL INPUT FRANGI ---
    # 1. Visible Image
    axes[0, 0].imshow(vis_data['visible'], cmap='gray')
    axes[0, 0].set_title("1. Visible Modality (Severe Gravel Clutter)", fontsize=14, fontweight='bold')
    
    # 2. Thermal Image
    axes[0, 1].imshow(vis_data['infrared'], cmap='gray')
    axes[0, 1].set_title("2. Decoded Inverted Thermal", fontsize=14, fontweight='bold')
    
    # 3. Ground Truth Mask
    axes[0, 2].imshow(vis_data['gt'], cmap='gray')
    axes[0, 2].set_title("3. Ground Truth (Fissure Target)", fontsize=14, fontweight='bold')
    
    # 4. Our Fused Multiscale Frangi
    axes[0, 3].imshow(vis_data['ours_fused_frangi'], cmap='magma')
    axes[0, 3].set_title("4. Ours: Fused Frangi Response (Hessian Level)", fontsize=14, fontweight='bold')
    
    # --- ROW 2: OURS (GENERALIZED FRANGI GRAPH ON GPU) ---
    # 5. Node Similarity Map
    axes[1, 0].imshow(vis_data['ours_similarity'], cmap='magma')
    axes[1, 0].set_title("5. Ours: Graph Node Similarities (Max)", fontsize=14, fontweight='bold')
    
    # 6. Graph Filtering Diagnostics
    h, w = vis_data['visible'].shape
    rgba_tau = np.zeros((h, w, 4))
    rgba_tau[vis_data['ours_tau_mask'] > 0] = [1.0, 1.0, 1.0, 0.3] # White for tau-filtered candidates
    rgba_comp = np.zeros((h, w, 4))
    rgba_comp[vis_data['ours_comp_mask'] > 0] = [0.0, 0.5, 1.0, 0.8] # Blue for large CCs
    axes[1, 1].imshow(np.zeros((h, w)), cmap='gray')
    axes[1, 1].imshow(rgba_tau)
    axes[1, 1].imshow(rgba_comp)
    axes[1, 1].set_title("6. Ours: Pruned Nodes (White) & Active CCs (Blue)", fontsize=14, fontweight='bold')
    
    # 7. Betweenness Centrality
    axes[1, 2].imshow(vis_data['ours_centrality'], cmap='hot')
    axes[1, 2].set_title("7. Ours: GPU Betweenness Centrality", fontsize=14, fontweight='bold')
    
    # 8. Ours final output
    axes[1, 3].imshow(vis_data['ours'], cmap='hot')
    axes[1, 3].set_title("8. Ours: Final Thickened Skeleton (5px)", fontsize=14, fontweight='bold')
    
    # --- ROW 3: BASELINE (STANDARD FRANGI + SAM 2) ---
    # 9. Baseline Standard Thermal Frangi
    axes[2, 0].imshow(vis_data['baseline_frangi'], cmap='magma')
    axes[2, 0].set_title("9. Baseline: Standard Thermal Frangi Response", fontsize=14, fontweight='bold')
    
    # 10. Top 0.5% Thresholded Mask
    axes[2, 1].imshow(vis_data['baseline_binary'], cmap='gray')
    axes[2, 1].set_title("10. Baseline: Top 0.5% Thresholded Mask", fontsize=14, fontweight='bold')
    
    # 11. Skeleton with 12 Prompt Points
    axes[2, 2].imshow(vis_data['baseline_skel'], cmap='gray')
    pts = vis_data['points']
    axes[2, 2].scatter(pts[:, 0], pts[:, 1], color='red', s=50, edgecolors='black', label='Sampled Prompts (N=12)')
    axes[2, 2].set_title("11. Baseline: Skeleton & Prompt Locations (Red)", fontsize=14, fontweight='bold')
    axes[2, 2].legend()
    
    # 12. Final Overlay comparison
    # White = Ground Truth, Green = Ours, Red = Baseline SAM 2
    rgba_gt = np.zeros((h, w, 4))
    rgba_gt[vis_data['gt'] > 0] = [1.0, 1.0, 1.0, 0.4] # White
    rgba_ours = np.zeros((h, w, 4))
    rgba_ours[vis_data['ours'] > 0] = [0.0, 1.0, 0.0, 0.4] # Green
    rgba_base = np.zeros((h, w, 4))
    rgba_base[vis_data['baseline'] > 0] = [1.0, 0.0, 0.0, 0.4] # Red
    
    axes[2, 3].imshow(vis_data['visible'], cmap='gray')
    axes[2, 3].imshow(rgba_gt)
    axes[2, 3].imshow(rgba_ours)
    axes[2, 3].imshow(rgba_base)
    axes[2, 3].set_title("12. Overlay (White: GT, Green: Ours, Red: SAM 2)", fontsize=14, fontweight='bold')
    
    for ax in axes.flat:
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
